
# GPT-2 Neutral (A) - Good (B) - Bad (D) output

**Edit the parameters in the next cell (marked _EDIT HERE_) and run the cells in order.**
- PROMPT: your input text prompt
- TEMPERATURE: try 0.4 (conservative), 0.6 (balanced), 0.8 (exploratory)
- SIM_THRESHOLD: 0.50 merges more (coarser), 0.60 splits more (finer)

`temperature` has to be a strictly positive float, otherwise your next token scores will be invalid. If you're looking for greedy decoding strategies (TEMPERATURE = 0.0), set `do_sample=False`

In [ ]:
# # GPT-2 Truth-Based Symbols **Mini-Lab**
#
# **Edit the parameters in the cell below (marked _EDIT HERE_) and run the cells in order.**
# - PROMPT: your input text prompt
# - TEMPERATURE: try 0.4 (conservative), 0.6 (balanced), 0.8 (exploratory)
# - Symbol assignments based on TRUTH VALUE:
#   - **A** = Neutral/Question/Ambiguous
#   - **B** = Correct/Reasonable
#   - **D** = Incorrect/Unreasonable
#
# `temperature` has to be a strictly positive float, otherwise your next token scores will be invalid. If you're looking for greedy decoding strategies (TEMPERATURE = 0.0), set `do_sample=False`

# In[1]:


# === EDIT HERE ===
PROMPT = "Maybe the earth is round. Maybe the earth is flat. Is the earth round or flat?"
# PROMPT = "Is the earth flat?"
MODEL = "gpt2"           # e.g., "gpt2", "distilgpt2"
TEMPERATURE = 0.0
TOP_K = 50
MAX_NEW_TOKENS = 150
SEED = 1

# Optional: input your own text instead of generating
INPUT_TEXT = ""  # paste your own text here (leave empty to generate with GPT-2)


In [ ]:

!pip -q install transformers==4.* torch scikit-learn matplotlib numpy


In [ ]:
import re, math, json, numpy as np, matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

# Updated regex for splitting sentences, including handling of newline characters
SENT_REGEX = re.compile(r'(?<=[.!?])\s+|\n+')
ABBREV_END = re.compile(r'(?:^|\s)(?:Dr|Mr|Ms|Mrs|Prof|Sr|Jr|St|Mt|No)\.$')

def split_sentences(text: str):
    # Split by punctuation followed by space or by newline characters
    parts = re.split(SENT_REGEX, text)
    merged = []
    for seg in parts:
        s = seg.strip()
        if not s: continue
        # Check for abbreviations at the end of the *last* merged sentence
        if merged and ABBREV_END.search(merged[-1]):
            merged[-1] += " " + s
        else:
            merged.append(s)
    return merged

def extract_question_subject(question):
    """
    Extract the core claim from a question to determine what 'yes' or 'no' would affirm.
    Returns: ('correct'|'incorrect'|'neutral', subject_description)
    """
    q_lower = question.lower()

    # Check for OR questions - these don't have a simple yes/no answer
    if ' or ' in q_lower:
        return ('neutral', 'or-question')

    # Check patterns in the ORIGINAL question (before removing question words)
    # This is important because "Is the earth flat?" contains "earth" and "flat"

    # Check for INCORRECT claims in questions
    incorrect_question_patterns = [
        # Earth/Astronomy - check for combinations
        ('earth', 'flat'),
        ('earth', 'not round'),
        ('planet', 'flat'),
        ('planet', 'not round'),
        ('world', 'flat'),
        ('world', 'not round'),
        # Vaccines
        ('vaccine', 'bad'),
        ('vaccine', 'dangerous'),
        ('vaccine', 'unsafe'),
        ('vaccine', 'autism'),
        # Science denial
        ('climate change', 'fake'),
        ('evolution', 'false'),
    ]

    for pattern_parts in incorrect_question_patterns:
        if all(part in q_lower for part in pattern_parts):
            return ('incorrect', f"question about {' '.join(pattern_parts)}")

    # Check for CORRECT claims in questions
    correct_question_patterns = [
        # Earth/Astronomy
        ('earth', 'round'),
        ('earth', 'not flat'),
        ('earth', 'spherical'),
        ('earth', 'sphere'),
        ('planet', 'round'),
        # Vaccines
        ('vaccine', 'safe'),
        ('vaccine', 'effective'),
        ('vaccine', 'work'),
        # Science
        ('climate change', 'real'),
        ('evolution', 'real'),
    ]

    for pattern_parts in correct_question_patterns:
        if all(part in q_lower for part in pattern_parts):
            return ('correct', f"question about {' '.join(pattern_parts)}")

    return ('neutral', None)

def classify_sentence_truth(sentence, previous_sentence=None):
    """
    Rule-based classifier for truth value with context awareness.
    Returns: 'A' (neutral), 'B' (correct), or 'D' (incorrect)

    Args:
        sentence: The sentence to classify
        previous_sentence: The sentence that came before (for context)
    """
    sentence_lower = sentence.lower()

    # PRIORITY: Handle context-dependent affirmations/negations FIRST
    # This must come before other pattern matching
    if previous_sentence and '?' in previous_sentence:
        question_type, subject = extract_question_subject(previous_sentence)

        # "The answer is yes" or "Yes" - affirms the question's claim
        if any(phrase in sentence_lower for phrase in ['answer is yes', 'yes,', 'yes.', 'yes!']):
            if question_type == 'correct':
                return 'B'  # Affirming a correct claim = correct
            elif question_type == 'incorrect':
                return 'D'  # Affirming an incorrect claim = incorrect
            # If neutral question, continue to other checks

        # "The answer is no" or "No" - negates the question's claim
        elif any(phrase in sentence_lower for phrase in ['answer is no', 'no,', 'no.', 'no!']):
            if question_type == 'correct':
                return 'D'  # Denying a correct claim = incorrect
            elif question_type == 'incorrect':
                return 'B'  # Denying an incorrect claim = correct
            # If neutral question, continue to other checks

    # Questions and neutral/ambiguous statements -> A
    if '?' in sentence:
        return 'A'
    if any(word in sentence_lower for word in ['maybe', 'might', 'could be', 'perhaps',
                                                 'possibly', 'unclear', 'uncertain']):
        return 'A'

    # Known CORRECT statements -> B
    correct_patterns = [
        # Earth/Astronomy
        'earth is round', 'earth is spherical', 'earth is a sphere', 'planet is round',
        'earth orbits', 'sun is a star', 'moon orbits earth',

        # Vaccines
        'vaccines are safe', 'vaccines are effective', 'vaccines work',
        'vaccines are good', 'vaccines are generally safe', 'vaccines prevent',
        'vaccination is safe', 'vaccination works', 'immunization is effective',

        # Basic science
        'water is h2o', 'gravity exists', 'evolution is real', 'climate change is real',
        'speed of light', 'photosynthesis', 'dna contains', 'cells divide',

        # NOTE: Removed 'answer is yes' from here - it's context-dependent
    ]

    if any(pattern in sentence_lower for pattern in correct_patterns):
        return 'B'

    # Known INCORRECT statements -> D
    incorrect_patterns = [
        # Earth/Astronomy
        'earth is flat', 'planet is flat', 'world is flat',
        'sun orbits earth', 'stars are small', 'moon is made of cheese',

        # Vaccines
        'vaccines are bad', 'vaccines are dangerous', 'vaccines are unsafe',
        'vaccines are generally unsafe', 'vaccines don\'t work', 'vaccines cause autism',
        'vaccination is dangerous', 'immunization is harmful',

        # Science denial
        'evolution is false', 'climate change is fake', 'gravity is a hoax',
        'dinosaurs never existed', 'moon landing was fake',

        # Basic misconceptions
        'humans use only 10%', 'lightning never strikes twice'
    ]

    if any(pattern in sentence_lower for pattern in incorrect_patterns):
        return 'D'

    # Default to neutral if uncertain
    return 'A'

def classify_by_truth(sentences):
    """
    Classify sentences by truth value with context awareness.
    Returns numeric labels: 0=A (neutral), 1=B (correct), 2=D (incorrect)
    """
    labels = []
    exemplars = {0: None, 1: None, 2: None}  # Store first example of each type
    classifications_detail = []  # For debugging

    for i, sentence in enumerate(sentences):
        # Get previous sentence for context
        previous_sentence = sentences[i-1] if i > 0 else None

        truth_label = classify_sentence_truth(sentence, previous_sentence)

        if truth_label == 'A':
            labels.append(0)
            if exemplars[0] is None:
                exemplars[0] = sentence
        elif truth_label == 'B':
            labels.append(1)
            if exemplars[1] is None:
                exemplars[1] = sentence
        elif truth_label == 'D':
            labels.append(2)
            if exemplars[2] is None:
                exemplars[2] = sentence
        else:
            labels.append(0)  # Default to neutral

        # Store detail for potential debugging
        classifications_detail.append({
            'index': i,
            'sentence': sentence,
            'previous': previous_sentence,
            'label': truth_label
        })

    return labels, exemplars, classifications_detail

def int_labels_to_truth_symbols(labels):
    """
    Convert numeric labels to truth-based symbols.
    0 -> A (neutral), 1 -> B (correct), 2 -> D (incorrect)
    """
    if not labels:
        return "", {}

    alphabet = {0: 'A', 1: 'B', 2: 'D'}
    sym = "".join(alphabet[i] for i in labels)
    return sym, alphabet

def run_length_encode(labels):
    if not labels: return []
    rle = []; cur = labels[0]; L = 1
    for x in labels[1:]:
        if x==cur: L+=1
        else: rle.append((cur,L)); cur=x; L=1
    rle.append((cur,L)); return rle

def classify_truth_dynamics(labels):
    """
    Analyze dynamics based on truth values.
    Focus on: settling to B (correct), D (incorrect), or oscillating
    """
    rle = run_length_encode(labels)
    if not rle:
        return {"class":"empty","switches":0,"mean_run":0.0,"settled_to":None}

    total = sum(L for _,L in rle)
    switches = max(0, len(rle)-1)
    mean_run = total/len(rle)

    # Count occurrences of each truth value
    counts = {0: 0, 1: 0, 2: 0}
    for s, L in rle:
        counts[s] = counts.get(s, 0) + L

    # Check what it settled to (if any)
    settled_to = None
    if len(rle) > 0 and rle[-1][1] >= 4:  # Last run is at least 4
        settled_to = rle[-1][0]

    # Classify behavior
    truth_map = {0: 'neutral', 1: 'correct', 2: 'incorrect'}

    if settled_to == 1:
        label = "converged_to_truth"
    elif settled_to == 2:
        label = "converged_to_falsehood"
    elif settled_to == 0:
        label = "settled_neutral"
    elif switches >= 6:
        label = "oscillatory"
    else:
        label = "mixed"

    return {
        "class": label,
        "switches": switches,
        "mean_run": mean_run,
        "settled_to": truth_map[settled_to] if settled_to is not None else None,
        "counts": {truth_map[k]: v for k, v in counts.items()}
    }

def time_to_settle_index(labels, min_suffix=3):
    n=len(labels)
    if n==0: return None
    last=labels[-1]; r=1; i=n-2
    while i>=0 and labels[i]==last:
        r+=1; i-=1
    if r>=min_suffix:
        return i+1
    return None

def plot_truth_barcode(labels, prompt_length, title="Truth-Based Symbol Stream"):
    if not labels:
        print("No labels to plot."); return

    # Define colors for truth values
    # 0 (A/neutral) = gray, 1 (B/correct) = green, 2 (D/incorrect) = red
    color_map = {
        0: '#808080',  # Gray for neutral
        1: '#2ecc71',  # Green for correct
        2: '#e74c3c'   # Red for incorrect
    }

    fig, ax = plt.subplots(figsize=(10, 1.5))
    for i, lab in enumerate(labels):
        ax.axvspan(i, i+1, color=color_map[lab], ec='white', linewidth=0.5)

    # Add vertical line to separate prompt from generated output
    if prompt_length > 0 and prompt_length < len(labels):
        ax.axvline(x=prompt_length, color='black', linewidth=2, linestyle='--', label='Prompt|Output boundary')

    ax.set_xlim(0, len(labels)); ax.set_ylim(0,1)
    ax.set_yticks([]); ax.set_xlabel("Sentence index"); ax.set_title(title)

    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#808080', label='A (Neutral/Question)'),
        Patch(facecolor='#2ecc71', label='B (Correct/Reasonable)'),
        Patch(facecolor='#e74c3c', label='D (Incorrect/Unreasonable)')
    ]
    if prompt_length > 0 and prompt_length < len(labels):
        from matplotlib.lines import Line2D
        legend_elements.append(Line2D([0], [0], color='black', linewidth=2, linestyle='--', label='Prompt|Output'))

    ax.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.15, 1))

    plt.tight_layout(); plt.show()



In [ ]:
# --- Generate or use your own text ---
if INPUT_TEXT.strip():
    text = INPUT_TEXT
    prompt_text = ""  # No prompt if using custom text
    print("=== Using your pasted INPUT_TEXT ===")
else:
    tok = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForCausalLM.from_pretrained(MODEL); model.eval()
    set_seed(SEED)
    inputs = tok(PROMPT, return_tensors="pt")
    out_ids = model.generate(
        **inputs, do_sample=False, temperature=TEMPERATURE, top_k=TOP_K,
        max_new_tokens=MAX_NEW_TOKENS, eos_token_id=tok.eos_token_id
    )
    text = tok.decode(out_ids[0], skip_special_tokens=True)
    prompt_text = PROMPT
    print("=== Original Prompt ===")
    print(f'"{PROMPT}"')
    print("\n=== Generated text (first 800 chars) ===")
    print(text[:800])



In [ ]:
# --- Sentences -> truth-based symbols ---
sentences = split_sentences(text)

# Count how many sentences are in the prompt
prompt_sentences = split_sentences(prompt_text) if prompt_text else []
prompt_length = len(prompt_sentences)

print(f"\nNumber of sentences total: {len(sentences)}")
print(f"Number of sentences in prompt: {prompt_length}")
print(f"Number of sentences generated: {len(sentences) - prompt_length}")

print("\n=== All Sentences with Classifications ===")
# Classify first to get labels
labels, exemplars, classifications_detail = classify_by_truth(sentences)
symbol_stream, alphabet = int_labels_to_truth_symbols(labels)

for i, s in enumerate(sentences[:30]):
    marker = "[PROMPT]" if i < prompt_length else "[OUTPUT]"
    label_str = ['A', 'B', 'D'][labels[i]]
    print(f"[{i:02d}] {marker} [{label_str}] {s}")
if len(sentences) > 30: print("... (truncated)")

# Split symbol stream at prompt boundary
if prompt_length > 0 and prompt_length < len(symbol_stream):
    prompt_symbols = symbol_stream[:prompt_length]
    output_symbols = symbol_stream[prompt_length:]
    symbol_stream_display = f"{prompt_symbols}-{output_symbols}"
else:
    symbol_stream_display = symbol_stream

print("\n=== Prompt Symbols ===")
if prompt_length > 0:
    print(symbol_stream[:prompt_length])
else:
    print("(No prompt - using custom INPUT_TEXT)")

print("\n=== Truth-Based Symbol Stream (Prompt-Output) ===")
print(symbol_stream_display)

print("\nLegend (symbol -> meaning and example):")
truth_meanings = {
    0: "A (Neutral/Question/Ambiguous)",
    1: "B (Correct/Reasonable)",
    2: "D (Incorrect/Unreasonable)"
}
for label_num, meaning in truth_meanings.items():
    if exemplars[label_num]:
        print(f"  {meaning}: \"{exemplars[label_num]}\"")


# In[5]:


# --- Truth dynamics analysis + visualization ---
info = classify_truth_dynamics(labels)
print("\n=== Truth-Based Dynamics Summary ===")
print(json.dumps(info, indent=2))

t_settle = time_to_settle_index(labels, min_suffix=3)
if t_settle is not None:
    settled_value = labels[-1]
    truth_map = {0: 'neutral', 1: 'correct', 2: 'incorrect'}
    print(f"\nTime to settle: {t_settle} (settled to {truth_map[settled_value]})")
else:
    print("\nTime to settle: did not settle (suffix >= 3).")

print("\n")
plot_truth_barcode(labels, prompt_length, title="Truth-Based Symbol Stream (Prompt | Output)")

print("\n=== INTERPRETATION GUIDE ===")
print("• Green (B): Model generated scientifically correct/reasonable statements")
print("• Red (D): Model generated incorrect/unreasonable statements")
print("• Gray (A): Model generated neutral/ambiguous/question statements")
print("• Dashed line: Separates prompt (left) from generated output (right)")
print("\n• Context-aware: 'Yes/No' answers are classified based on preceding question")
print("• 'converged_to_truth': Model settled on correct statements")
print("• 'converged_to_falsehood': Model settled on incorrect statements")
print("• 'oscillatory': Model keeps switching between truth values")

print("\n=== EXPERIMENTS TO TRY ===")
print("1) Try: 'Is the earth round? The answer is yes.' vs 'Is the earth flat? The answer is yes.'")
print("2) Change PROMPT to controversial topics (vaccines, climate, etc.)")
print("3) Try different TEMPERATURE values (0.0, 0.5, 0.8) and compare")
print("4) Observe if model converges to truth (B) or falsehood (D)")
print("5) Add your own correct/incorrect patterns to classify_sentence_truth()")

In [ ]:
# --- Export dynamic network for Gephi ---
import pandas as pd

# Create transition edges (only for generated output, not prompt)
edges = []
symbol_map = {0: 'A', 1: 'B', 2: 'D'}

# Start from the last prompt symbol to first output symbol
if prompt_length > 0 and prompt_length < len(labels):
    # Transitions in generated output only
    for i in range(prompt_length, len(labels)):
        if i == prompt_length:
            # First transition: from last prompt to first output
            source = symbol_map[labels[i-1]]
            target = symbol_map[labels[i]]
        else:
            # Subsequent transitions within output
            source = symbol_map[labels[i-1]]
            target = symbol_map[labels[i]]

        timestep = i - prompt_length + 1  # Start timestep at 1 for first output
        edges.append({
            'Source': source,
            'Target': target,
            'Timestep': timestep,
            'Timestep_End': timestep + 1,
            'Weight': 1
        })
else:
    # No prompt, use all transitions
    for i in range(1, len(labels)):
        source = symbol_map[labels[i-1]]
        target = symbol_map[labels[i]]
        edges.append({
            'Source': source,
            'Target': target,
            'Timestep': i,
            'Timestep_End': i + 1,
            'Weight': 1
        })

# Create edge DataFrame
edges_df = pd.DataFrame(edges)

# Create nodes DataFrame
nodes = []
for symbol_id, symbol_name in symbol_map.items():
    truth_meanings_full = {
        0: "Neutral/Question/Ambiguous",
        1: "Correct/Reasonable",
        2: "Incorrect/Unreasonable"
    }
    nodes.append({
        'Id': symbol_name,
        'Label': symbol_name,
        'Type': truth_meanings_full[symbol_id]
    })

nodes_df = pd.DataFrame(nodes)

# Save to CSV files
edges_df.to_csv('gephi_edges.csv', index=False)
nodes_df.to_csv('gephi_nodes.csv', index=False)

print("\n=== GEPHI EXPORT ===")
print(f"✓ Created gephi_nodes.csv with {len(nodes_df)} nodes")
print(f"✓ Created gephi_edges.csv with {len(edges_df)} edges")
print("\n=== Edge Preview (first 10 transitions) ===")
print(edges_df.head(10).to_string(index=False))

print("\n=== Transition Summary ===")
transition_counts = edges_df.groupby(['Source', 'Target']).size().reset_index(name='Count')
transition_counts = transition_counts.sort_values('Count', ascending=False)
print(transition_counts.to_string(index=False))

print("\n=== HOW TO USE IN GEPHI ===")
print("1. Open Gephi")
print("2. Go to File > Import Spreadsheet")
print("3. Import 'gephi_nodes.csv' as 'Nodes table'")
print("4. Import 'gephi_edges.csv' as 'Edges table'")
print("5. In the Timeline panel, enable timeline")
print("6. Configure timeline to use 'Timestep' as start and 'Timestep_End' as end")
print("7. Use Layout algorithms (e.g., Force Atlas 2) to visualize")
print("8. Color nodes by 'Type' to distinguish A (neutral), B (correct), D (incorrect)")
print("\nNote: Each edge appears at 'Timestep' and disappears at 'Timestep_End'")
print("This creates a dynamic network where edges appear and disappear over time")


In [ ]:
# --- Dynamic Network Visualization in Python ---
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Prepare transitions from generated output only
transitions = []
symbol_map = {0: 'A', 1: 'B', 2: 'D'}

if prompt_length > 0 and prompt_length < len(labels):
    for i in range(prompt_length, len(labels)):
        source = symbol_map[labels[i-1]]
        target = symbol_map[labels[i]]
        timestep = i - prompt_length + 1
        transitions.append({'source': source, 'target': target, 'timestep': timestep})
else:
    for i in range(1, len(labels)):
        source = symbol_map[labels[i-1]]
        target = symbol_map[labels[i]]
        transitions.append({'source': source, 'target': target, 'timestep': i})

# Node positions (fixed layout - triangle)
node_positions = {
    'A': (0, 1),      # Top
    'B': (-0.8, -0.5),  # Bottom left
    'D': (0.8, -0.5)   # Bottom right
}

# Node colors
node_colors = {
    'A': '#808080',  # Gray - Neutral
    'B': '#2ecc71',  # Green - Correct
    'D': '#e74c3c'   # Red - Incorrect
}

# Create static plot showing cumulative network
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# LEFT: Static cumulative network
ax1.set_xlim(-1.2, 1.2)
ax1.set_ylim(-1, 1.5)
ax1.set_aspect('equal')
ax1.axis('off')
ax1.set_title('Cumulative Transition Network', fontsize=14, fontweight='bold')

# Draw nodes
for node, pos in node_positions.items():
    circle = plt.Circle(pos, 0.15, color=node_colors[node], ec='black', linewidth=2, zorder=3)
    ax1.add_patch(circle)
    ax1.text(pos[0], pos[1], node, ha='center', va='center',
             fontsize=20, fontweight='bold', color='white', zorder=4)

# Count all transitions for cumulative view
from collections import Counter
transition_counter = Counter()
for trans in transitions:
    edge = (trans['source'], trans['target'])
    transition_counter[edge] += 1

# Draw edges with thickness based on frequency
for (source, target), count in transition_counter.items():
    start_pos = node_positions[source]
    end_pos = node_positions[target]

    # Self-loop
    if source == target:
        # Draw circular arc for self-loop
        angle = {'A': 90, 'B': 210, 'D': 330}[source]
        loop = mpatches.FancyBboxPatch(
            (start_pos[0]-0.15, start_pos[1]+0.15), 0.3, 0.3,
            boxstyle="round,pad=0.05",
            linewidth=1 + count*0.5,
            edgecolor=node_colors[source],
            facecolor='none',
            alpha=0.7
        )
        ax1.add_patch(loop)
        # Add count label
        label_pos = (start_pos[0], start_pos[1] + 0.35)
        ax1.text(label_pos[0], label_pos[1], f'×{count}',
                ha='center', va='center', fontsize=10, fontweight='bold')
    else:
        # Draw arrow
        dx = end_pos[0] - start_pos[0]
        dy = end_pos[1] - start_pos[1]
        length = (dx**2 + dy**2)**0.5
        # Shorten to not overlap with nodes
        start_x = start_pos[0] + 0.15 * dx / length
        start_y = start_pos[1] + 0.15 * dy / length
        end_x = end_pos[0] - 0.15 * dx / length
        end_y = end_pos[1] - 0.15 * dy / length

        ax1.arrow(start_x, start_y, end_x - start_x, end_y - start_y,
                 head_width=0.1, head_length=0.1,
                 linewidth=1 + count*0.5,
                 fc=node_colors[target], ec=node_colors[target],
                 alpha=0.7, length_includes_head=True, zorder=2)

        # Add count label at midpoint
        mid_x = (start_x + end_x) / 2
        mid_y = (start_y + end_y) / 2
        ax1.text(mid_x, mid_y, f'×{count}',
                ha='center', va='center', fontsize=10, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

# Add legend
legend_elements = [
    mpatches.Patch(facecolor='#808080', label='A: Neutral/Question'),
    mpatches.Patch(facecolor='#2ecc71', label='B: Correct/Reasonable'),
    mpatches.Patch(facecolor='#e74c3c', label='D: Incorrect/Unreasonable')
]
ax1.legend(handles=legend_elements, loc='upper left', fontsize=10)

# RIGHT: Single timestep view (will show last transition as static)
ax2.set_xlim(-1.2, 1.2)
ax2.set_ylim(-1, 1.5)
ax2.set_aspect('equal')
ax2.axis('off')

if transitions:
    last_trans = transitions[-1]
    ax2.set_title(f'Last Transition (t={last_trans["timestep"]}): {last_trans["source"]} → {last_trans["target"]}',
                 fontsize=14, fontweight='bold')

    # Draw nodes
    for node, pos in node_positions.items():
        circle = plt.Circle(pos, 0.15, color=node_colors[node], ec='black', linewidth=2, zorder=3)
        ax2.add_patch(circle)
        ax2.text(pos[0], pos[1], node, ha='center', va='center',
                fontsize=20, fontweight='bold', color='white', zorder=4)

    # Draw only the active edge
    source = last_trans['source']
    target = last_trans['target']
    start_pos = node_positions[source]
    end_pos = node_positions[target]

    if source == target:
        # Self-loop
        loop = mpatches.FancyBboxPatch(
            (start_pos[0]-0.15, start_pos[1]+0.15), 0.3, 0.3,
            boxstyle="round,pad=0.05",
            linewidth=3,
            edgecolor=node_colors[source],
            facecolor='none'
        )
        ax2.add_patch(loop)
    else:
        # Arrow
        dx = end_pos[0] - start_pos[0]
        dy = end_pos[1] - start_pos[1]
        length = (dx**2 + dy**2)**0.5
        start_x = start_pos[0] + 0.15 * dx / length
        start_y = start_pos[1] + 0.15 * dy / length
        end_x = end_pos[0] - 0.15 * dx / length
        end_y = end_pos[1] - 0.15 * dy / length

        ax2.arrow(start_x, start_y, end_x - start_x, end_y - start_y,
                 head_width=0.2, head_length=0.2,
                 linewidth=3,
                 fc=node_colors[target], ec=node_colors[target],
                 length_includes_head=True, zorder=2)

plt.tight_layout()
plt.show()

# Print transition sequence
print("\n=== DYNAMIC NETWORK TRANSITION SEQUENCE ===")
print("Timestep | Transition | Description")
print("-" * 50)
for trans in transitions[:20]:  # Show first 20
    t = trans['timestep']
    s = trans['source']
    tgt = trans['target']
    if s == tgt:
        desc = f"Model stuck in {s} (repeating same type)"
    else:
        desc = f"Model switched from {s} to {tgt}"
    print(f"t={t:3d}    | {s} → {tgt}   | {desc}")
if len(transitions) > 20:
    print(f"... ({len(transitions) - 20} more transitions)")

print(f"\nTotal transitions: {len(transitions)}")

In [ ]:
# --- Dynamic Network Animation in Python (Colab-Compatible) ---
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
import numpy as np
import time

# Prepare transitions from generated output only
transitions = []
symbol_map = {0: 'A', 1: 'B', 2: 'D'}

if prompt_length > 0 and prompt_length < len(labels):
    for i in range(prompt_length, len(labels)):
        source = symbol_map[labels[i-1]]
        target = symbol_map[labels[i]]
        timestep = i - prompt_length + 1
        transitions.append({'source': source, 'target': target, 'timestep': timestep})
else:
    for i in range(1, len(labels)):
        source = symbol_map[labels[i-1]]
        target = symbol_map[labels[i]]
        transitions.append({'source': source, 'target': target, 'timestep': i})

# Node positions (fixed layout - triangle)
node_positions = {
    'A': (0, 1),      # Top
    'B': (-0.8, -0.5),  # Bottom left
    'D': (0.8, -0.5)   # Bottom right
}

# Node colors
node_colors = {
    'A': '#808080',  # Gray - Neutral
    'B': '#2ecc71',  # Green - Correct
    'D': '#e74c3c'   # Red - Incorrect
}

print(f"\n=== CREATING DYNAMIC NETWORK ANIMATION ===")
print(f"Total timesteps: {len(transitions)}")

def draw_frame(frame_idx, transitions):
    """Draw a single frame of the animation"""
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1, 1.5)
    ax.set_aspect('equal')
    ax.axis('off')

    # Draw nodes
    for node, pos in node_positions.items():
        circle = plt.Circle(pos, 0.15, color=node_colors[node], ec='black', linewidth=2, zorder=3)
        ax.add_patch(circle)
        ax.text(pos[0], pos[1], node, ha='center', va='center',
                fontsize=24, fontweight='bold', color='white', zorder=4)

    # Draw edge for current frame
    if frame_idx < len(transitions):
        trans = transitions[frame_idx]
        source = trans['source']
        target = trans['target']
        timestep = trans['timestep']

        start_pos = node_positions[source]
        end_pos = node_positions[target]

        if source == target:
            # Self-loop
            loop = mpatches.FancyBboxPatch(
                (start_pos[0]-0.15, start_pos[1]+0.15), 0.3, 0.3,
                boxstyle="round,pad=0.05",
                linewidth=4,
                edgecolor=node_colors[source],
                facecolor='none',
                zorder=2
            )
            ax.add_patch(loop)
        else:
            # Arrow between nodes
            dx = end_pos[0] - start_pos[0]
            dy = end_pos[1] - start_pos[1]
            length = (dx**2 + dy**2)**0.5
            start_x = start_pos[0] + 0.15 * dx / length
            start_y = start_pos[1] + 0.15 * dy / length
            end_x = end_pos[0] - 0.20 * dx / length
            end_y = end_pos[1] - 0.20 * dy / length

            arrow = mpatches.FancyArrowPatch(
                (start_x, start_y), (end_x, end_y),
                arrowstyle='->,head_width=0.4,head_length=0.4',
                linewidth=4,
                color=node_colors[target],
                zorder=2
            )
            ax.add_patch(arrow)

        ax.set_title(f'Timestep {timestep}: {source} → {target}',
                    fontsize=16, fontweight='bold', pad=20)

    # Add legend
    legend_elements = [
        mpatches.Patch(facecolor='#808080', label='A: Neutral'),
        mpatches.Patch(facecolor='#2ecc71', label='B: Correct'),
        mpatches.Patch(facecolor='#e74c3c', label='D: Incorrect')
    ]
    ax.legend(handles=legend_elements, loc='upper left', fontsize=10)

    return fig

# Create and save animation as GIF
num_frames = min(30, len(transitions))  # Limit to 30 frames for reasonable file size
print(f"Creating GIF animation with {num_frames} frames...")

fig, ax = plt.subplots(figsize=(8, 8))

def init():
    ax.clear()
    return []

def animate(frame):
    ax.clear()
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1, 1.5)
    ax.set_aspect('equal')
    ax.axis('off')

    # Draw nodes
    for node, pos in node_positions.items():
        circle = plt.Circle(pos, 0.15, color=node_colors[node], ec='black', linewidth=2, zorder=3)
        ax.add_patch(circle)
        ax.text(pos[0], pos[1], node, ha='center', va='center',
                fontsize=24, fontweight='bold', color='white', zorder=4)

    # Draw edge for current frame
    if frame < len(transitions):
        trans = transitions[frame]
        source = trans['source']
        target = trans['target']
        timestep = trans['timestep']

        start_pos = node_positions[source]
        end_pos = node_positions[target]

        if source == target:
            # Self-loop with arrow
            # Calculate position for self-loop
            theta = {'A': 90, 'B': 210, 'D': -30}[source]  # Angle for loop position
            theta_rad = np.radians(theta)
            loop_radius = 0.25

            # Loop goes out and comes back
            loop_start_x = start_pos[0] + 0.15 * np.cos(theta_rad - 0.5)
            loop_start_y = start_pos[1] + 0.15 * np.sin(theta_rad - 0.5)
            loop_end_x = start_pos[0] + 0.15 * np.cos(theta_rad + 0.5)
            loop_end_y = start_pos[1] + 0.15 * np.sin(theta_rad + 0.5)

            arrow = mpatches.FancyArrowPatch(
                (loop_start_x, loop_start_y),
                (loop_end_x, loop_end_y),
                arrowstyle='->,head_width=0.3,head_length=0.3',
                connectionstyle=f"arc3,rad=1.5",
                linewidth=4,
                edgecolor=node_colors[source],
                facecolor=node_colors[source],
                zorder=2,
                mutation_scale=20
            )
            ax.add_patch(arrow)
        else:
            dx = end_pos[0] - start_pos[0]
            dy = end_pos[1] - start_pos[1]
            length = (dx**2 + dy**2)**0.5
            start_x = start_pos[0] + 0.18 * dx / length
            start_y = start_pos[1] + 0.18 * dy / length
            end_x = end_pos[0] - 0.18 * dx / length
            end_y = end_pos[1] - 0.18 * dy / length

            arrow = mpatches.FancyArrowPatch(
                (start_x, start_y), (end_x, end_y),
                arrowstyle='->',
                linewidth=4,
                edgecolor=node_colors[target],
                facecolor=node_colors[target],
                zorder=2,
                mutation_scale=30
            )
            ax.add_patch(arrow)

        ax.set_title(f'Timestep {timestep}: {source} → {target}',
                    fontsize=16, fontweight='bold', pad=20)

    legend_elements = [
        mpatches.Patch(facecolor='#808080', label='A: Neutral'),
        mpatches.Patch(facecolor='#2ecc71', label='B: Correct'),
        mpatches.Patch(facecolor='#e74c3c', label='D: Incorrect')
    ]
    ax.legend(handles=legend_elements, loc='upper left', fontsize=10)

    return []

anim = FuncAnimation(fig, animate, init_func=init, frames=num_frames,
                    interval=500, repeat=True, blit=False)

# Save as GIF
gif_filename = 'network_animation.gif'
print("Saving animation as GIF (this may take a moment)...")
anim.save(gif_filename, writer=PillowWriter(fps=2))
plt.close()

print(f"✓ Animation saved as '{gif_filename}'")
print("\nDisplaying animation...")

# Display the GIF in Colab
display(Image(filename=gif_filename))

print("\n=== TRANSITION SEQUENCE ===")
print("Timestep | Transition | Description")
print("-" * 50)
for trans in transitions[:20]:  # Show first 20
    t = trans['timestep']
    s = trans['source']
    tgt = trans['target']
    if s == tgt:
        desc = f"Stuck in {s} state"
    else:
        desc = f"Switched from {s} to {tgt}"
    print(f"t={t:3d}    | {s} → {tgt}   | {desc}")
if len(transitions) > 20:
    print(f"... ({len(transitions) - 20} more transitions)")

print(f"\nTotal transitions: {len(transitions)}")
print(f"\n💡 The animation loops automatically. Each frame shows one transition.")
print(f"💡 The GIF file '{gif_filename}' has been saved and can be downloaded.")
